# Project 01 — Exploratory Data Analysis (EDA) on Titanic
**Track:** Data Science & AI

## Objective
Inspect, clean and analyze Titanic passenger data using Pandas. Compare survival rates by sex, passenger class and age band.

## Questions
1. Did female passengers have a higher survival rate than male passengers?
2. Did passenger class relate to survival rate?
3. Did survival rate differ across passenger age bands?

**Dataset:** `titanic.csv` in this repository. Keep the CSV in the same folder as this notebook.

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
data_file = Path("titanic.csv")
if not data_file.exists():
    raise FileNotFoundError("Upload titanic.csv to the notebook's working folder.")

raw_df = pd.read_csv(data_file)
df = raw_df.rename(columns={
    "Survived": "survived",
    "Pclass": "pclass",
    "Sex": "sex",
    "Age": "age",
    "Cabin": "deck",
    "Embarked": "embarked"
}).copy()
df["sex"] = df["sex"].str.lower()
df["embarked"] = df["embarked"].str.upper()

print("Dataset shape:", df.shape)

## 2. Inspect the DataFrame
Use `head()`, `info()`, and `describe()` to understand the records, data types and summary statistics.

In [ ]:
display(df.head())
df.info()
display(df.describe(include="all").T)
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False))
print("Duplicate rows before cleaning:", df.duplicated().sum())

## 3. Handle missing values and duplicates

- **Age:** fill missing values with the median age for the same passenger class and sex group. The median is less affected by extreme values than the mean.
- **Embarked:** fill missing values with the most common port (mode), suitable for the small number of missing entries.
- **Deck:** remove the column because most values are missing, making it unsuitable for reliable comparisons in this analysis.
- **Duplicates:** count exact duplicate rows. Do not delete rows blindly; retain them unless there is a clear reason to remove them.

In [ ]:
missing_before = df.isna().sum()
print("Missing values before cleaning:")
display(missing_before[missing_before > 0])

df["age"] = df["age"].fillna(
    df.groupby(["pclass", "sex"])["age"].transform("median")
)
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"])

duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows after cleaning:", duplicate_count)
print("Missing values after cleaning:")
display(df.isna().sum())

## 4. Check data types
`survived` is binary (0 = did not survive, 1 = survived). Passenger class is an ordinal numeric variable; sex and embarkation port are categorical.

In [ ]:
df["survived"] = df["survived"].astype("int64")
df["pclass"] = df["pclass"].astype("int64")
df["sex"] = df["sex"].astype("category")
df["embarked"] = df["embarked"].astype("category")
print(df.dtypes)
print("Survival values:", sorted(df["survived"].unique()))

## 5. Group and compare survival rates
The mean of the binary `survived` column represents the survival rate. Counts are included to provide group-size context.

In [ ]:
survival_by_sex = df.groupby("sex", observed=True)["survived"].agg(
    survival_rate="mean", passengers="count"
).sort_values("survival_rate", ascending=False)

survival_by_class = df.groupby("pclass")["survived"].agg(
    survival_rate="mean", passengers="count"
).sort_values("survival_rate", ascending=False)

df["age_band"] = pd.cut(
    df["age"], bins=[0, 12, 18, 40, 60, 100],
    labels=["Child", "Teen", "Adult", "Middle-aged", "Senior"],
    include_lowest=True
)
survival_by_age = df.groupby("age_band", observed=True)["survived"].agg(
    survival_rate="mean", passengers="count"
).sort_values("survival_rate", ascending=False)

print("Survival by sex")
display(survival_by_sex.style.format({"survival_rate": "{:.1%}"}))
print("Survival by passenger class")
display(survival_by_class.style.format({"survival_rate": "{:.1%}"}))
print("Survival by age band")
display(survival_by_age.style.format({"survival_rate": "{:.1%}"}))

## 6. Visualize the comparisons
Bar heights show survival percentages, not passenger counts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
(survival_by_sex["survival_rate"] * 100).plot(kind="bar", ax=axes[0], title="Survival by sex")
(survival_by_class["survival_rate"] * 100).sort_index().plot(kind="bar", ax=axes[1], title="Survival by class")
(survival_by_age["survival_rate"] * 100).plot(kind="bar", ax=axes[2], title="Survival by age band")

for ax in axes:
    ax.set_ylabel("Survival rate (%)")
    ax.set_xlabel("")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## 7. Answer the three questions with numerical findings
This cell generates the answers from the cleaned DataFrame so the figures remain consistent with the analysis.

In [ ]:
female_rate = survival_by_sex.loc["female", "survival_rate"]
male_rate = survival_by_sex.loc["male", "survival_rate"]
class_rates = survival_by_class["survival_rate"]
age_rates = survival_by_age["survival_rate"]

print(f"1. Sex: female passengers survived at {female_rate:.1%}, compared with {male_rate:.1%} for male passengers.")
print(f"2. Class: class {int(class_rates.idxmax())} had the highest survival rate ({class_rates.max():.1%}); class {int(class_rates.idxmin())} had the lowest ({class_rates.min():.1%}).")
print(f"3. Age: {age_rates.idxmax()} had the highest age-band survival rate ({age_rates.max():.1%}); {age_rates.idxmin()} had the lowest ({age_rates.min():.1%}).")
print(f"Overall survival rate: {df['survived'].mean():.1%}")

## 8. Conclusion, limitations and submission checks

The descriptive analysis shows differences in survival rates by sex, passenger class and age band. Female and first-class passengers had higher observed survival rates. These patterns are associations in this dataset and do not prove causation.

**Limitations:** Age-band results partly depend on the age-imputation method. The Senior group is small, so its percentage is less stable. Results describe this historical dataset and should not be generalized without further study.

**Final checklist**
- [x] Three questions stated at the beginning
- [x] `head()`, `info()`, and `describe()` used
- [x] Missing values counted and treated with reasons
- [x] Duplicate rows counted; no blind deletion
- [x] Data types checked
- [x] Survival grouped by sex, class and age band
- [x] Numerical answers and charts included
- [ ] Restart the kernel and run all cells in your own environment
- [ ] Export notebook as HTML/PDF if required by your course
- [ ] Share the repository/notebook link on InternCircle My Tasks

**Before submitting:** Run the notebook from top to bottom with `titanic.csv` in the working folder. Add your name/student details if required.